In [ ]:
from copy import copy
from io import BytesIO
from math import *
from build123d import *
from build123d.topology.shape_core import Shape, downcast
from ocp_vscode import *
import subprocess
from pathlib import Path

set_port(3939)
set_viewer_config(port=3939)



PCB = "../pcb/esp-butt/esp-butt.kicad_pcb"
PCB_NAME = "esp-butt"
PCB_EXPORT_DIR = "../pcb/export"
PCB_STEP = Path(PCB_EXPORT_DIR) / "esp-butt.step"
PCB_POS = Path(PCB_EXPORT_DIR) / "pos.csv"
EXPORT_DIR = "./export"

# For use mostly with "Run All Cells", to avoid accidentally exporting when not intended.
EXPORT = True


def kicad_pcb_export(export_format, output, *args):
  subprocess.run(
    [KICAD_CLI, "pcb", "export", export_format, "--output", output, *args, PCB],
    check=True,
  )


def fast_copy(shape: Shape) -> Shape:
  cls = shape.__class__
  result = cls.__new__(cls)
  for key, value in shape.__dict__.items():
    if key == "_wrapped":
      result._wrapped = downcast(shape.wrapped.Located(shape.wrapped.Location()))
    elif key in ("_reset_tok", "_python_frame"):
      pass
    else:
      try:
        setattr(result, key, value if key.startswith("_NodeMixin__") else copy(value))
      except Exception as e:
        print(f"Error copying attribute '{key}': {e}")
  return result


def copy_located(shape: Shape, location=Location()):
  shape = fast_copy(shape)
  shape.location = location
  return shape


def export(shape: Shape):
  if EXPORT:
    if not shape.label:
      raise ValueError("Shape must have a label to be exported.")
    export_dir = Path(EXPORT_DIR)
    export_dir.mkdir(parents=True, exist_ok=True)

    version = 1
    previous_file = None

    for existing_file in export_dir.glob(f"{shape.label}.v*.step"):
      existing_version = int(existing_file.stem.split(".v")[-1])
      if existing_version >= version:
        previous_file = existing_file
        version = existing_version + 1

    prev_data = previous_file.read_bytes() if previous_file else None
    new_data = BytesIO()
    export_step(shape, new_data)

    if prev_data != new_data.getvalue():
      new_file = export_dir / f"{shape.label}.v{version}"
      new_file.write_bytes(new_data.getvalue())
      print(f"Exported '{shape.label}' to {new_file} (version {version})")

In [11]:
kicad_pcb_export(
  "step",
  PCB_STEP,
  "--force",
  "--subst-models",
  "--include-silkscreen",
)
kicad_pcb_export(
  "pos",
  PCB_POS,
  "--side",
  "both",
  "--format",
  "csv",
  "--units",
  "mm",
)

Determining PCB data.
Build STEP data.
Adding component S2.
Adding component R7.
Adding component J2.
Adding component U1.
Adding component J1.
Adding component J1.
Adding component R8.
Adding component S1.
Adding component C2.
Adding component R2.
Adding component R4.
Adding component R10.
Adding component R9.
Adding component C1.
Adding component R3.
Adding component R5.
Adding component R1.
Adding component R6.
Adding component C3.
Create PCB solid model.
Board outline: found 96 initial points.
Build board outlines (1 outlines) with 96 points.
Build board cutouts and holes (154 hole(s)).
Subtracting holes for shapes
Subtracting holes for pads
Subtracting holes for vias
Generate board full shape.
Writing STEP file.

*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******   

In [12]:
import csv
from anytree import LevelOrderIter

full_pcb = import_step(PCB_STEP)


def pcb_rename_from_pos():
  pos_map: dict[Vector, str] = {}
  with open(PCB_POS, newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
      pos_map[Vector(float(row["PosX"]), float(row["PosY"]))] = f"{row['Ref']} {row['Val']}"

  children = [c for c in full_pcb.children if not c.label.startswith(f"{PCB_NAME}_")]
  while children and pos_map:
    (pos, label) = pos_map.popitem()
    if closest := next((c for c in children if (c.location.position - pos).length < 2), None):
      closest.label = label
      children.remove(closest)
    elif closest := next((c for c in children if (c.location.position - pos).length < 10), None):
      closest.label = label
      children.remove(closest)


def pcb_get(name) -> Shape:
  for c in full_pcb.children:
    if c.label == name:
      return c
  raise ValueError(f"Top level node with label '{name}' not found")


def pcb_find(prefix, copy=True) -> Shape:
  for c in LevelOrderIter(full_pcb):
    if c.label.startswith(prefix):
      if copy:
        c = fast_copy(c)
        c.locate(c.global_location)
      return c
  raise ValueError(f"Node with label starting with '{prefix}' not found")


def pcb_rename(old, new):
  if node := pcb_get(old):
    node.label = new

def pcb_color(label, color):
  if node := pcb_find(label, copy=False):
    node.color = color

pcb_rename_from_pos()
pcb_rename("kicad_embedded_9C69275F55A1B470E08F556AC7123E48", "J1 Screen")

reset_show()
show_object(full_pcb)


---------------ccccccccccccccccccccccccccccccccccccccccccc


# Case

In [13]:
reset_show()

pcb_color('J1 Screen', Color("black"))
pcb_color('R7', Color("silver"))
pcb_color('R8', Color("silver"))


pcb = pcb_get(f'{PCB_NAME}_PCB')
pcb_bottom_face = pcb.faces().group_by(Axis.Z)[0].first
pcb_outer = pcb_bottom_face.outer_wire()
pcb_size_x = pcb_outer.bounding_box().size.X
pcb_size_y = pcb_outer.bounding_box().size.Y

pcb_usb = pcb_find('USB_TYPE_C_PORT')
pcb_usb_outer = pcb_usb.faces().group_by(Axis.Y)[-1].first.outer_wire()

pcb_switch = pcb_find('S2')
pcb_switch_face = pcb_switch.faces().group_by(Axis.Y)[-1].first

pcb_screen = pcb_find("J1 Screen")
pcb_screen_screen = pcb_screen.faces().filter_by(Plane.XY).group_by(Axis.Z)[-1].sort_by(SortBy.AREA).last


pcb_encoder = pcb_find("S1")
pcb_sliders = [pcb_find("R7"), pcb_find("R8")]

show_object(pcb_outer)
# show_object(pcb_usb)
show_object(pcb_usb_outer)
# show_object(pcb_switch)
show_object(pcb_switch_face)
# show_object(pcb_screen)
show_object(pcb_screen_screen)




+
c+


In [5]:
reset_show()

pcb_lip = 1.0
wall_thickness = 1.25
wall_offset = 0.1
basement_height = 5.0

bottom_thickness = 1.0
top_thickness = 1.0
usb_cutout_extra = 0.1
switch_cutout_r = 2.5
lid_interface = wall_thickness * 2
lid_gap = 0.1
cutout_extra = 0.25

antenna_width = 37.6
antenna_height = 17.7
antenna_depth = 0.5

lid_wedge_size = 10

main_height = pcb_screen.bounding_box().max.Z


def lid_wedge(mode=Mode.ADD, size=lid_wedge_size):
  """Create a 45 degree wedge. Its "bottom" is centered in x_dir going towards +z, and its "point" is towards +y."""
  w = size
  h = lid_interface
  Wedge(
    xsize=w,
    ysize=h,
    zsize=h,
    xmin=h / 2,
    zmin=h / 2,
    xmax=w - h / 2,
    zmax=h / 2,
    align=(Align.CENTER, Align.MIN, Align.MIN),
    mode=mode,
  )


def pcb_outer_offset_sketch(offset_amount, z=None):
  with BuildSketch(Plane.XY.offset(z) if z else Plane.XY, mode=Mode.PRIVATE) as s:
    offset(pcb_outer, amount=offset_amount)
    make_face()
  return s.sketch


with BuildPart() as case_bottom:
  extrude(
    pcb_outer_offset_sketch(wall_offset + wall_thickness),
    amount=-bottom_thickness - basement_height,
  )
  extrude(
    pcb_outer_offset_sketch(wall_offset + wall_thickness),
    amount=main_height,
  )
  # cut basement cavity
  extrude(
    pcb_outer_offset_sketch(wall_offset - pcb_lip),
    amount=-basement_height,
    mode=Mode.SUBTRACT,
  )
  # cut main cavity
  extrude(
    pcb_outer_offset_sketch(wall_offset),
    amount=main_height,
    mode=Mode.SUBTRACT,
  )

  inner_y_faces = faces(Select.LAST).filter_by(Plane.XZ)

  wedge_planes = []
  for f in inner_y_faces:
    for u, x in [(0, lid_wedge_size / 2), (0.5, 0), (1, -lid_wedge_size / 2)]:
      l = f.location_at(u, 0)
      p = Plane(l)
      p.origin += Vector(x * p.x_dir.X, 0, 0)

      wedge_planes.append(Plane(p.origin, x_dir=p.x_dir, y_dir=-p.z_dir))

  with Locations(wedge_planes):
    lid_wedge()

  back = faces().sort_by(Axis.Y).last

  with BuildSketch(back) as s:
    l = project(pcb_usb_outer, mode=Mode.PRIVATE)
    make_face(l)
    o = offset(faces(), amount=0.1, side=Side.RIGHT)
  extrude(amount=-wall_thickness, mode=Mode.SUBTRACT)

  with BuildSketch(pcb_switch_face.project_to_shape(back, Axis.Y.direction)) as s:
    Circle(switch_cutout_r)
  extrude(amount=-wall_thickness, mode=Mode.SUBTRACT)

  top: Face = faces().filter_by(Plane.XY).sort_by(Axis.Z)[-1]
  top_left_edge_inner: Edge = (
    top.edges().filter_by(Plane.YZ).filter_by(GeomType.LINE).sort_by(Axis.X)[1]
  )

  antenna = top_left_edge_inner.end_point() + Vector(-wall_thickness / 2, -antenna_width / 2, 0)
  with BuildSketch(Plane(antenna)) as s:
    Rectangle(antenna_depth, antenna_width)
  extrude(amount=-antenna_height, mode=Mode.SUBTRACT)
  with BuildSketch(Plane(antenna)) as s:
    Rectangle(wall_thickness, antenna_width / 3, align=(Align.MIN, Align.CENTER))
  extrude(amount=-antenna_height / 3 * 2, mode=Mode.SUBTRACT)


with BuildPart() as case_top:
  extrude(
    pcb_outer_offset_sketch(wall_offset + wall_thickness, z=main_height),
    amount=top_thickness,
  )
  lip_full = extrude(
    pcb_outer_offset_sketch(wall_offset - lid_gap, z=main_height),
    amount=-lid_interface,
  )
  extrude(
    pcb_outer_offset_sketch(wall_offset - lid_gap - wall_thickness, z=main_height),
    amount=-lid_interface,
    mode=Mode.SUBTRACT,
  )

  chamfer(
    faces(Select.LAST).filter_by(Plane.XY).sort_by(SortBy.AREA).last.outer_wire().edges(),
    length=wall_thickness/2,
  )

  bounding_box(pcb_screen, mode=Mode.SUBTRACT)
  extrude(
    project(
      pcb_screen_screen,
      workplane=Plane(faces().sort_by(SortBy.AREA)[-2]),
      mode=Mode.PRIVATE,
    )
    .faces()
    .sort_by(Axis.Z)
    .first,
    amount=-wall_thickness,
    taper=-45,
    mode=Mode.SUBTRACT,
  )


  outer_y_faces = lip_full.faces().filter_by(Plane.XZ)

  wedge_planes = []
  for f in inner_y_faces:
    for u, x in [(0, lid_wedge_size / 2), (0.5, 0), (1, -lid_wedge_size / 2)]:
      l = f.location_at(u, 0)
      p = Plane(l)
      p.origin += Vector(x * p.x_dir.X, 0, 0)

      wedge_planes.append(Plane(p.origin, x_dir=p.x_dir, y_dir=-p.z_dir))

  with Locations(wedge_planes):
    with Locations((0, wall_thickness, 0)):
      lid_wedge()
    lid_wedge(mode=Mode.SUBTRACT)

  

  top = faces().sort_by(SortBy.AREA).last
  for slider in pcb_sliders:
    slider_bbox = slider.bounding_box()
    slider_center_plane = Plane(slider_bbox.center(), x_dir=(1, 0, 0), y_dir=(0, 0, 1))
    section: Face = slider.intersect(top)[0]
    c1: Location = section.center_location
    c2 = c1.mirror(slider_center_plane)
    radius = section.bounding_box().size.X + cutout_extra
    slot_location = Plane((c1.position + c2.position) / 2, x_dir=(0, 1, 0), y_dir=(1, 1, 0))
    length = (
      Edge.make_line(c1.position, c2.position).length
      + section.bounding_box().size.Y
      - radius
      + cutout_extra * 2
    )
    with BuildSketch(slot_location) as s:
      SlotCenterToCenter(length, radius)
    extrude(amount=wall_thickness, mode=Mode.SUBTRACT)

  # Encoder cutout - find the circular shaft cross-section at the top face
  encoder_section = pcb_encoder.intersect(top)
  encoder_face = encoder_section.faces().sort_by(SortBy.AREA).last
  encoder_circle = encoder_face.edges().filter_by(GeomType.CIRCLE).first.geom_adaptor().Circle()
  encoder_center = Vector(encoder_circle.Position().Location())
  encoder_radius = encoder_circle.Radius() + cutout_extra
  with BuildSketch(Plane(encoder_center)) as s:
    Circle(encoder_radius)
  extrude(amount=-wall_thickness, mode=Mode.SUBTRACT)


# # show_object(full_pcb)
show_object(pcb_screen)
# show_object(pcb_encoder)
# show_object(pcb_sliders[0])
# show_object(pcb_sliders[1])

case_bottom = case_bottom.part
case_bottom.label = "case_bottom"
case_bottom.color = Color(0.1, 0.1, 0.1)

case_top = case_top.part
case_top.label = "case_top"
case_top.color = Color(0.1, 0.1, 0.1)

show_object(case_bottom)
show_object(case_top)
export(case_bottom)
export(case_top)

+
c+
c++

# Slider knob

In [14]:
width = 25
height = 9.5
depth = 12
divot_depth = 3.5
grove_count = 7
grove_fillet = 0.2
grove_depth = 1.25
center_channel_width = 1.25
center_insert_depth = 1.25
center_insert_tip_size = (0.7, 0.4)
tip_fillet = 1
outer_fillet = 0.1
bottom_chamfer = 0.5
lever_width = 8
lever_thickness = 1.2
lever_slot_width = 3.0  # Neck width of DP lever
lever_slot_depth = 1.0  # Thickness of lever neck
lever_grip_depth = 4  # How deep the slot goes into the knob

reset_show()

with BuildPart() as slider_knob_body:
  with BuildSketch(Plane.XY) as s:
    hw = width / 2
    hw16 = hw * (1 / 6)
    hw56 = hw * (5 / 6)
    with BuildLine():
      # Bottom line
      b = Line((-hw56, 0), (hw56, 0))
      # 45 degree lines
      rl = Line(b @ 1, (hw, hw16))
      ll = Line(b @ 0, (-hw, hw16))
      # Side arcs
      ra = RadiusArc((hw56, height), rl @ 1, radius=height)
      la = RadiusArc(ll @ 1, (-hw56, height), radius=height)
      # Divot arc
      SagittaArc(ra @ 0, la @ 1, divot_depth)
    make_face()
    divot_arc = s.edges()[3]
    with Locations(
      [
        Location(divot_arc.position_at(p), (0, 0, divot_arc.tangent_angle_at(p)))
        for p in [(i + 1) / (grove_count + 1) for i in range(grove_count) if i != grove_count // 2]
      ]
    ):
      Circle(grove_depth / 2, mode=Mode.SUBTRACT)

    tip_vertices = vertices().filter_by(lambda v: isclose(v.Y, height))
    grove_vertices = (
      vertices()
      .filter_by(lambda v: v not in tip_vertices)
      .filter_by_position(Axis.Y, height - divot_depth, height)
    )
    fillet(grove_vertices, grove_fillet)
    fillet(tip_vertices, tip_fillet)

  extrude(amount=depth)

  chamfer(edges().group_by(Axis.Y)[0].group_by(Axis.X)[1], bottom_chamfer)

  center = slider_knob_body.part.intersect(Plane.ZY.offset(-center_channel_width / 2))
  with BuildSketch(Plane.ZY.offset(-center_channel_width / 2)) as c:
    offset(
      center.edges().group_by(Axis.Y)[2:4],
      amount=center_insert_depth,
      side=Side.RIGHT,
    )
    make_face()
  extrude(c.sketch.rotate(Axis.X, 180), amount=center_channel_width, mode=Mode.SUBTRACT)

  with BuildPart(mode=Mode.PRIVATE) as slider_knob_insert:
    with BuildSketch(Plane.ZY.offset(-center_channel_width / 2)) as c:
      i = offset(center.edges().group_by(Axis.Y)[2:4], amount=center_insert_depth, side=Side.RIGHT)
      t = i.vertices().group_by(Axis.Y)[0].sort_by_distance(i.center())[0]
      with Locations([Location((-t.Z, -t.Y), (0, 180, 0))]):
        t = Triangle(
          a=center_insert_tip_size[1],
          b=center_insert_tip_size[0],
          C=90,
          align=(Align.MAX, Align.MAX),
          mode=Mode.ADD,
        )
      mirror(t, Plane.YZ.offset(-i.center().Z))
      make_face()
    extrude(c.sketch.rotate(Axis.X, 180), amount=center_channel_width)

  outer = list(filter(lambda e: not (e.geom_type == GeomType.LINE), [
    *edges().group_by(Axis.Z)[0].sort_by(Axis.Y)[3:],
    *edges().group_by(Axis.Z)[-1].sort_by(Axis.Y)[3:],
  ]))
  fillet(outer, outer_fillet)

  with BuildSketch(faces().group_by(Axis.Y)[0]) as lever_hole:
    Rectangle(lever_slot_width, lever_slot_depth)
  extrude(amount=-lever_grip_depth, mode=Mode.SUBTRACT)

slider_knob_body = slider_knob_body.part
slider_knob_body.label = "slider_knob_body"
slider_knob_body.color = Color(0.5, 0.1, 0.1)

slider_knob_insert = slider_knob_insert.part
slider_knob_insert.label = "slider_knob_insert"
slider_knob_insert.color = Color(0.1, 0.1, 0.1)

slider_knob = Compound(label="slider_knob", children=[slider_knob_body, slider_knob_insert])

show_object(slider_knob_body)
show_object(slider_knob_insert)
export(slider_knob_body)
export(slider_knob_insert)

c
cc


Exported 'slider_knob_body' to export/slider_knob_body.v1.step (version 1)
Exported 'slider_knob_insert' to export/slider_knob_insert.v1.step (version 1)


In [7]:
full_assembly = Compound(
  label="esp-butt",
  children=(
    copy_located(full_pcb),
    copy_located(case_bottom),
    copy_located(case_top),
    copy_located(slider_knob, Location((61, -70, 18), (90, 90, 0))),
    copy_located(slider_knob, Location((112, -70, 18), (90, 90, 0))),
  ),
)

full_assembly.locate(Location((-pcb.center().X, -pcb.center().Y, 0)))


reset_show()
show_object(full_assembly)

-----------------++++++++++++++++++++++++++++++++++++++++++++++c


# Encoder knob


In [54]:
radius = 10
height = 15
chamfer_length = 2
shaft_radius = 3
shaft_flat_length = 5.2
shaft_insert_depth = 10

reset_show()

with BuildPart() as encoder_knob:
  Cylinder(radius=radius, height=height, align=(Align.CENTER, Align.CENTER, Align.MIN))
  with BuildSketch() as s:
    h = sqrt(shaft_radius**2 - (shaft_flat_length / 2) ** 2)
    with BuildLine():
      l = Line((-shaft_flat_length / 2, -h), (shaft_flat_length / 2, -h))
      RadiusArc(l @ 1, l @ 0, radius=shaft_radius, short_sagitta=False)
    make_face()
  extrude(amount=shaft_insert_depth, mode=Mode.SUBTRACT)
  # chamfer top
  chamfer(edges().sort_by(Axis.Z).last, length=chamfer_length)

  knurl_count = 20
  knurl_pitch = height * 1.5
  knurl_depth = 0.4

  helix_template_cw = Edge.make_helix(knurl_pitch, height, radius)
  helix_template_ccw = Edge.make_helix(knurl_pitch, height, radius, lefthand=True)

  for i in range(knurl_count):
    angle = i * 360 / knurl_count
    for helix in [
      helix_template_cw.rotate(Axis.Z, angle),
      helix_template_ccw.rotate(Axis.Z, angle),
    ]:
      with BuildSketch(Plane(helix.position_at(0), z_dir=helix.tangent_at(0))):
        RegularPolygon(knurl_depth, 3)
      sweep(path=helix, mode=Mode.SUBTRACT)


show_object(encoder_knob)
show_object(pcb_encoder)


KeyboardInterrupt: 

In [13]:
radius = 10
height = 15
chamfer_length = 2
shaft_radius = 3
shaft_flat_length = 5.2
shaft_insert_depth = 10

reset_show()


def knurl_helix(pitch, height, radius, tip_count, offset=0, left=False):
  return [
    Edge.make_helix(pitch, height, radius, lefthand=left).rotate(
      Axis.Z, (i + offset) * 360 / tip_count
    )
    for i in range(tip_count)
  ]


def knurl_edges(
  inside_edges: list[Wire], outside_edges: list[Wire], position_at: float, tip_count: int
):
  for i in range(tip_count):
    yield Edge.make_line(
      inside_edges[i].position_at(position_at),
      outside_edges[i].position_at(position_at),
    )
    yield Edge.make_line(
      outside_edges[i].position_at(position_at),
      inside_edges[(i + 1) % tip_count].position_at(position_at),
    )


def knurl_faces(
  inside_edges: list[Wire],
  outside_edges: list[Wire],
  bottom_edges: list[Edge],
  top_edges: list[Edge],
  tip_count: int,
):
  for i in range(tip_count):
    yield Face.make_surface(
      [
        inside_edges[i],
        outside_edges[i],
        bottom_edges[2 * i],
        top_edges[2 * i],
      ],
    )
    yield Face.make_surface(
      [
        outside_edges[i],
        inside_edges[(i + 1) % tip_count],
        bottom_edges[2 * i + 1],
        top_edges[2 * i + 1],
      ],
    )


def make_knurled_faces(radius, height, depth, pitch, tip_count, left=False):
  inside_edges = knurl_helix(pitch, height, radius - depth, tip_count, 0, left)
  outside_edges = knurl_helix(pitch, height, radius, tip_count, 0.5, left)
  bottom_edges = list(knurl_edges(inside_edges, outside_edges, 0, tip_count))
  top_edges = list(knurl_edges(inside_edges, outside_edges, 1, tip_count))
  outside_faces = list(knurl_faces(inside_edges, outside_edges, bottom_edges, top_edges, tip_count))
  return bottom_edges, top_edges, outside_faces

def make_knurled_cylinder(radius, height, depth, pitch, tip_count, left=False):
  bottom_edges_r, top_edges_r, outside_faces_r = make_knurled_faces(radius, height, depth, pitch, tip_count, left=left)
  # show_object(outside_faces_r)
  
  
  # bottom_edges2, top_edges2, outside_faces2 = make_knurled_faces(radius, height, depth, pitch, tip_count, left=True)
  inner = Solid.make_cylinder(radius - depth, height)
  inner_face = inner.faces().filter_by(GeomType.CYLINDER).first

  show_object(Wire(bottom_edges_r))
  show_object(inner.edges().sort_by(Axis.Z).first)

  bottom_face_r = Face(Wire(bottom_edges_r), [Wire(inner.edges().sort_by(Axis.Z).first)])
  top_face_r = Face(Wire(top_edges_r), [Wire(inner.edges().sort_by(Axis.Z).last)])
  
  # # outside_faces2 = [f.translate(Vector(0, 0, height/tip_count)) for f in outside_faces2]
  # # top_face = Face(Wire(top_edges2)).translate(Vector(0, 0, height/tip_count))
  # show_object(outside_faces)
  # show_object(outside_faces2)
  # show_object(bottom_face)
  # show_object(top_face)

  
  show_object(bottom_face_r)
  show_object(top_face_r)
  return Solid(Shell(outside_faces_r + [inner_face, bottom_face_r, top_face_r]))

# def make_knurled_cylinder(radius, height, depth, pitch, tip_count):
#     inner_r = radius - depth
#     half = height / 2

#     bottom_edges, mid_edges_r, outer_faces_r = make_knurled_faces(radius, half, depth, pitch, tip_count, left=False)
#     mid_edges_l, top_edges, outer_faces_l = make_knurled_faces(radius, half, depth, pitch, tip_count, left=True)
#     # Translate upper half
#     outer_faces_l = [f.translate(Vector(0, 0, half)) for f in outer_faces_l]
#     top_edges = [e.translate(Vector(0, 0, half)) for e in top_edges]

#     bottom_face = Face(Wire(bottom_edges), [Wire([Edge.make_circle(inner_r)])])
#     top_face = Face(Wire(top_edges), [Wire([Edge.make_circle(inner_r, Plane.XY.offset(height))])])
#     mid_face = Face.extrude(Edge.make_circle(inner_r, Plane.XY.offset(half)), (0, 0, half))  # annular ring at junction
#     inner_face = Face.extrude(Edge.make_circle(inner_r), (0, 0, height))

#     show_object(outer_faces_r)
#     show_object(outer_faces_l)

#     show_object(bottom_face)
#     show_object(top_face)
#     show_object(mid_face)
#     show_object(inner_face)
#     # shell = Shell([*outer_faces_r, *outer_faces_l, bottom_face, top_face, inner_face])
#     # return Solid(shell).fuse(Solid.make_cylinder(inner_r, height))

knurl_count = 20
knurl_depth = 0.4
knurl_height = height - chamfer_length
knurl_pitch = knurl_height * 1.5  # ~1 helix turn over the grip zone

# reset_show()

# with BuildPart() as encoder_knob:
#   add(make_knurled_cylinder(radius, knurl_height, knurl_depth, knurl_pitch, knurl_count))
#   add(make_knurled_cylinder(radius, knurl_height, knurl_depth, knurl_pitch, knurl_count, left=True), mode=Mode.SUBTRACT)
#   # Smooth top band — gets the chamfer
#   # with Locations(Location((0, 0, knurl_height))):
#   #   Cylinder(radius, chamfer_length, align=(Align.CENTER, Align.CENTER, Align.MIN))
#   # # D-shaped shaft hole
#   # with BuildSketch() as s:
#   #   h = sqrt(shaft_radius**2 - (shaft_flat_length / 2) ** 2)
#   #   with BuildLine():
#   #     l = Line((-shaft_flat_length / 2, -h), (shaft_flat_length / 2, -h))
#   #     RadiusArc(l @ 1, l @ 0, radius=shaft_radius, short_sagitta=False)
#   #   make_face()
#   # extrude(amount=shaft_insert_depth, mode=Mode.SUBTRACT)
#   # # chamfer(edges().sort_by(Axis.Z).last, length=chamfer_length)

# encoder_knob = encoder_knob.part
# # encoder_knob.label = "encoder_knob"
# # encoder_knob.color = Color(0.5, 0.1, 0.1)

# show_object(encoder_knob)
# show_object(pcb_encoder)

In [25]:
radius = 10
height = 15
chamfer_length = 2
shaft_radius = 3
shaft_flat_length = 5.2
shaft_insert_depth = 10

knurl_count = 20
knurl_depth = 0.4
knurl_height = height - chamfer_length
knurl_pitch = knurl_height * 1.5  # ~1 helix turn over the grip zone

slices = 20

reset_show()

with BuildPart() as encoder_knob_slice:
  with BuildSketch() as s:
    with BuildLine():
      c = CenterArc((0, 0), radius, 0, 360/slices)
      Line((0,0), c @ 0)
      Line((0,0), c @ 1)
    make_face()
  extrude(amount=height)

  # hi = Edge.make_helix(knurl_pitch/slices, height/slices, (radius-knurl_depth)/slices)
  # ho = Edge.make_helix(knurl_pitch/slices, height/slices, radius/slices).rotate(Axis.Z, -0.5*360/knurl_count)
  # ht = Edge.make_helix(knurl_pitch/slices, height/slices, (radius-knurl_depth)/slices).rotate(Axis.Z, -1.5*360/knurl_count)
  show_object(encoder_knob_slice.part)
  show_object(Edge.make_helix(15, height/slices, radius))
  show_object(Edge.make_helix(15, height/slices, radius-knurl_depth).rotate(Axis.Z, -0.5*360/knurl_count))
  show_object(Edge.make_helix(15, height/slices, radius-knurl_depth).rotate(Axis.Z, -0.5*360/knurl_count))
  show_object(Edge.make_helix(15, height/slices, radius-knurl_depth).rotate(Axis.Z, -0.5*360/knurl_count))
  show_object(Edge.make_helix(15, height/slices, radius-knurl_depth).rotate(Axis.Z, -0.5*360/knurl_count))
  # show_object(ho)
  # show_object(ht)

encoder_knob_slices = [
  copy_located(encoder_knob_slice.part, Location((0,0,0), (0, 0, i * 360/slices)))
  for i in range(slices)
]

encoder_knob = encoder_knob_slices[0].fuse(*encoder_knob_slices[1:], glue=True)

show_object(encoder_knob)

c
c
c
cc


In [27]:
radius = 10
height = 15
chamfer_length = 2
shaft_radius = 3
shaft_flat_length = 5.2
shaft_insert_depth = 10

reset_show()


def knurl_helix(pitch, height, radius, tip_count, offset=0, left=False):
  return [
    Edge.make_helix(pitch, height, radius, lefthand=left).rotate(
      Axis.Z, (i + offset) * 360 / tip_count
    )
    for i in range(tip_count)
  ]


def knurl_edges(
  inside_edges: list[Wire], outside_edges: list[Wire], position_at: float, tip_count: int
):
  for i in range(tip_count):
    yield Edge.make_line(
      inside_edges[i].position_at(position_at),
      outside_edges[i].position_at(position_at),
    )
    yield Edge.make_line(
      outside_edges[i].position_at(position_at),
      inside_edges[(i + 1) % tip_count].position_at(position_at),
    )


def knurl_faces(
  inside_edges: list[Wire],
  outside_edges: list[Wire],
  bottom_edges: list[Edge],
  top_edges: list[Edge],
  tip_count: int,
):
  for i in range(tip_count):
    yield Face.make_surface(
      [
        inside_edges[i],
        outside_edges[i],
        bottom_edges[2 * i],
        top_edges[2 * i],
      ],
    )
    yield Face.make_surface(
      [
        outside_edges[i],
        inside_edges[(i + 1) % tip_count],
        bottom_edges[2 * i + 1],
        top_edges[2 * i + 1],
      ],
    )


def make_knurled_faces(radius, height, depth, pitch, tip_count, left=False):
  inside_edges = knurl_helix(pitch, height, radius - depth, tip_count, 0, left)
  outside_edges = knurl_helix(pitch, height, radius, tip_count, 0.5, left)
  bottom_edges = list(knurl_edges(inside_edges, outside_edges, 0, tip_count))
  top_edges = list(knurl_edges(inside_edges, outside_edges, 1, tip_count))
  outside_faces = list(knurl_faces(inside_edges, outside_edges, bottom_edges, top_edges, tip_count))
  return bottom_edges, top_edges, outside_faces

def make_knurled_cylinder(radius, height, depth, pitch, tip_count, left=False):
  bottom_edges_r, top_edges_r, outside_faces_r = make_knurled_faces(radius, height, depth, pitch, tip_count, left=left)
  # show_object(outside_faces_r)
  
  
  # bottom_edges2, top_edges2, outside_faces2 = make_knurled_faces(radius, height, depth, pitch, tip_count, left=True)
  inner = Solid.make_cylinder(radius - depth, height)
  inner_face = inner.faces().filter_by(GeomType.CYLINDER).first

  show_object(Wire(bottom_edges_r))
  show_object(inner.edges().sort_by(Axis.Z).first)

  bottom_face_r = Face(Wire(bottom_edges_r), [Wire(inner.edges().sort_by(Axis.Z).first)])
  top_face_r = Face(Wire(top_edges_r), [Wire(inner.edges().sort_by(Axis.Z).last)])
  
  # # outside_faces2 = [f.translate(Vector(0, 0, height/tip_count)) for f in outside_faces2]
  # # top_face = Face(Wire(top_edges2)).translate(Vector(0, 0, height/tip_count))
  # show_object(outside_faces)
  # show_object(outside_faces2)
  # show_object(bottom_face)
  # show_object(top_face)

  
  show_object(bottom_face_r)
  show_object(top_face_r)
  return Solid(Shell(outside_faces_r + [inner_face, bottom_face_r, top_face_r]))

# def make_knurled_cylinder(radius, height, depth, pitch, tip_count):
#     inner_r = radius - depth
#     half = height / 2

#     bottom_edges, mid_edges_r, outer_faces_r = make_knurled_faces(radius, half, depth, pitch, tip_count, left=False)
#     mid_edges_l, top_edges, outer_faces_l = make_knurled_faces(radius, half, depth, pitch, tip_count, left=True)
#     # Translate upper half
#     outer_faces_l = [f.translate(Vector(0, 0, half)) for f in outer_faces_l]
#     top_edges = [e.translate(Vector(0, 0, half)) for e in top_edges]

#     bottom_face = Face(Wire(bottom_edges), [Wire([Edge.make_circle(inner_r)])])
#     top_face = Face(Wire(top_edges), [Wire([Edge.make_circle(inner_r, Plane.XY.offset(height))])])
#     mid_face = Face.extrude(Edge.make_circle(inner_r, Plane.XY.offset(half)), (0, 0, half))  # annular ring at junction
#     inner_face = Face.extrude(Edge.make_circle(inner_r), (0, 0, height))

#     show_object(outer_faces_r)
#     show_object(outer_faces_l)

#     show_object(bottom_face)
#     show_object(top_face)
#     show_object(mid_face)
#     show_object(inner_face)
#     # shell = Shell([*outer_faces_r, *outer_faces_l, bottom_face, top_face, inner_face])
#     # return Solid(shell).fuse(Solid.make_cylinder(inner_r, height))

knurl_count = 20
knurl_depth = 0.4
knurl_height = 1
knurl_pitch = knurl_height * 1.5  # ~1 helix turn over the grip zone

# reset_show()

with BuildPart() as encoder_knob:
  add(make_knurled_cylinder(radius, knurl_height, knurl_depth, knurl_pitch, knurl_count))
#   add(make_knurled_cylinder(radius, knurl_height, knurl_depth, knurl_pitch, knurl_count, left=True), mode=Mode.SUBTRACT)
#   # Smooth top band — gets the chamfer
#   # with Locations(Location((0, 0, knurl_height))):
#   #   Cylinder(radius, chamfer_length, align=(Align.CENTER, Align.CENTER, Align.MIN))
#   # # D-shaped shaft hole
#   # with BuildSketch() as s:
#   #   h = sqrt(shaft_radius**2 - (shaft_flat_length / 2) ** 2)
#   #   with BuildLine():
#   #     l = Line((-shaft_flat_length / 2, -h), (shaft_flat_length / 2, -h))
#   #     RadiusArc(l @ 1, l @ 0, radius=shaft_radius, short_sagitta=False)
#   #   make_face()
#   # extrude(amount=shaft_insert_depth, mode=Mode.SUBTRACT)
#   # # chamfer(edges().sort_by(Axis.Z).last, length=chamfer_length)

encoder_knob = encoder_knob.part
# # encoder_knob.label = "encoder_knob"
# # encoder_knob.color = Color(0.5, 0.1, 0.1)

show_object(encoder_knob)
# show_object(pcb_encoder)



+


++
c++
